# 🛒 Shopper Spectrum

## Customer Segmentation and Product Recommendation System

**Author:** Aditya Raj

**Technologies Used:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-Learn · K-Means Clustering · RFM Analysis · Collaborative Filtering

**Objective:** To segment customers based on purchasing behavior and recommend products using transaction data.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from mpl_toolkits.mplot3d import Axes3D
import joblib, os, warnings
warnings.filterwarnings("ignore")

print("All libraries loaded!")
print("Pandas version:", pd.__version__)

## Step 2 — Dataset Collection and Understanding

This dataset contains retail transaction records including:

| Column | Description |
|---|---|
| InvoiceNo | Transaction number |
| StockCode | Unique product code |
| Description | Name of the product |
| Quantity | Number of products purchased |
| InvoiceDate | Date and time of transaction |
| UnitPrice | Price per product |
| CustomerID | Unique identifier for each customer |
| Country | Country of the customer |

In [ ]:
# Load dataset
df = pd.read_csv("online_retail.csv", encoding='ISO-8859-1')
print("Shape:", df.shape)
df.head()

In [ ]:
# Check data types and structure
df.info()

In [ ]:
# Missing values analysis
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}))

print("\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== UNIQUE COUNTS ===")
print("Unique Invoices  :", df['InvoiceNo'].nunique())
print("Unique Customers :", df['CustomerID'].nunique())
print("Unique Products  :", df['Description'].nunique())
print("Unique Countries :", df['Country'].nunique())

### Dataset Observations

- Dataset contains 541,909 transaction-level records
- CustomerID contains 135,080 missing values (24.93%)
- Description has 1,454 missing values
- 5,268 duplicate rows exist
- InvoiceDate needs to be parsed as datetime
- Data cleaning is required before modeling

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Prepare EDA copy
df_eda = df.copy()
df_eda['InvoiceDate'] = pd.to_datetime(df_eda['InvoiceDate'])
df_eda['TotalAmount'] = df_eda['Quantity'] * df_eda['UnitPrice']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Exploratory Data Analysis — Overview', fontsize=15, fontweight='bold')

# Top 10 countries
top_countries = df_eda['Country'].value_counts().head(10)
axes[0,0].barh(top_countries.index[::-1], top_countries.values[::-1], color='steelblue')
axes[0,0].set_title('Top 10 Countries by Transactions')
axes[0,0].set_xlabel('Number of Transactions')

# Top 10 products
top_products = df_eda.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(10)
axes[0,1].barh(top_products.index[::-1], top_products.values[::-1], color='coral')
axes[0,1].set_title('Top 10 Best-Selling Products')
axes[0,1].set_xlabel('Total Quantity Sold')

# Monthly revenue
df_eda['Month'] = df_eda['InvoiceDate'].dt.to_period('M')
monthly_sales = df_eda.groupby('Month')['TotalAmount'].sum()
axes[1,0].plot(monthly_sales.index.astype(str), monthly_sales.values, marker='o', color='green')
axes[1,0].set_title('Monthly Revenue Trend')
axes[1,0].set_xlabel('Month'); axes[1,0].set_ylabel('Revenue (£)')
axes[1,0].tick_params(axis='x', rotation=45)

# Sales distribution
cap = df_eda['TotalAmount'].quantile(0.95)
axes[1,1].hist(df_eda[df_eda['TotalAmount'] < cap]['TotalAmount'], bins=50, color='purple', edgecolor='white')
axes[1,1].set_title('Sales Amount Distribution (95th percentile)')
axes[1,1].set_xlabel('Transaction Amount (£)'); axes[1,1].set_ylabel('Count')

plt.tight_layout()
plt.show()

### EDA Insight

- United Kingdom dominates transactions by a large margin
- A small group of products drives the majority of sales volume
- Monthly revenue shows seasonal peaks (Q4 holiday period)
- Most transactions are small-value; a few customers drive high revenue

In [ ]:
# Day-of-week and hourly patterns
df_eda['DayOfWeek'] = df_eda['InvoiceDate'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_sales = df_eda.groupby('DayOfWeek')['TotalAmount'].sum().reindex(day_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Purchase Pattern Analysis', fontsize=13, fontweight='bold')

axes[0].bar(day_sales.index, day_sales.values, color='royalblue', edgecolor='white')
axes[0].set_title('Revenue by Day of Week')
axes[0].set_xlabel('Day'); axes[0].set_ylabel('Total Revenue (£)')
axes[0].tick_params(axis='x', rotation=30)

df_eda['Hour'] = df_eda['InvoiceDate'].dt.hour
hourly = df_eda.groupby('Hour')['TotalAmount'].sum()
axes[1].plot(hourly.index, hourly.values, marker='o', color='teal')
axes[1].set_title('Revenue by Hour of Day')
axes[1].set_xlabel('Hour (24h)'); axes[1].set_ylabel('Total Revenue (£)')

plt.tight_layout()
plt.show()

### Pattern Insight

- Most purchases happen on weekdays — Thursday has the highest volume
- Peak purchase hours are between 10am and 3pm
- Weekends show lower activity — this can guide email campaign timing

## Step 4 — Data Cleaning

| Issue | Action |
|---|---|
| Missing CustomerID (24.93%) | Drop rows |
| Cancelled invoices (InvoiceNo starts with 'C') | Remove |
| Negative / zero Quantity | Remove |
| Zero / negative UnitPrice | Remove |
| Duplicate rows (5,268) | Drop duplicates |
| Missing Description | Drop rows |

In [ ]:
df_clean = df.copy()
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Step 1 — Remove missing CustomerID
before = len(df_clean)
df_clean = df_clean.dropna(subset=['CustomerID'])
print(f"Removed {before - len(df_clean):,} rows — missing CustomerID")

# Step 2 — Remove cancelled invoices
before = len(df_clean)
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]
print(f"Removed {before - len(df_clean):,} rows — cancelled invoices")

# Step 3 — Remove invalid Quantity
before = len(df_clean)
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"Removed {before - len(df_clean):,} rows — invalid Quantity")

# Step 4 — Remove invalid UnitPrice
before = len(df_clean)
df_clean = df_clean[df_clean['UnitPrice'] > 0]
print(f"Removed {before - len(df_clean):,} rows — invalid UnitPrice")

# Step 5 — Drop duplicates
before = len(df_clean)
df_clean = df_clean.drop_duplicates()
print(f"Removed {before - len(df_clean):,} rows — duplicates")

# Step 6 — Drop missing Description
before = len(df_clean)
df_clean = df_clean.dropna(subset=['Description'])
print(f"Removed {before - len(df_clean):,} rows — missing Description")

# Step 7 — Fix CustomerID type
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

# Step 8 — Create TotalAmount
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"\n✅ Clean dataset shape: {df_clean.shape}")
print(f"   Total rows removed : {len(df) - len(df_clean):,} ({(1 - len(df_clean)/len(df))*100:.1f}%)")
df_clean.head()

### Cleaning Result — Dataset is clean and ready for modeling

## Step 5 — Feature Engineering: RFM Analysis

**RFM** measures three key customer dimensions:
- **Recency** — How recently did the customer purchase? (lower = better)
- **Frequency** — How many unique transactions? (higher = better)
- **Monetary** — Total money spent? (higher = better)

In [ ]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date.date()}")

rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate',  lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',    'nunique'),
    Monetary  = ('TotalAmount',  'sum')
).reset_index()

print(f"\nRFM table shape: {rfm.shape}")
rfm.describe().round(2)

In [ ]:
# RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RFM Feature Distributions', fontsize=13, fontweight='bold')

for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'], ['tomato','royalblue','gold']):
    cap = rfm[col].quantile(0.95)
    ax.hist(rfm[rfm[col] < cap][col], bins=40, color=color, edgecolor='white')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col); ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### RFM Interpretation

- Recency is right-skewed — many customers purchased recently
- Frequency is concentrated at low values — most customers buy 1–3 times
- Monetary is heavily skewed — a few customers drive large revenue

## Step 6 — Standardize RFM Values

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print("Scaled shape:", rfm_scaled.shape)
print("Mean (should be ~0):", rfm_scaled.mean(axis=0).round(4))
print("Std  (should be ~1):", rfm_scaled.std(axis=0).round(4))

## Step 7 — Finding Optimal Number of Clusters

In [ ]:
inertia, sil_scores = [], []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))
    print(f"k={k}  |  Inertia={km.inertia_:.0f}  |  Silhouette={sil_scores[-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Choosing Optimal Number of Clusters', fontsize=13, fontweight='bold')

axes[0].plot(list(K_range), inertia, marker='o', color='steelblue')
axes[0].axvline(x=4, color='red', linestyle='--', label='k=4 chosen')
axes[0].set_title('Elbow Curve')
axes[0].set_xlabel('Number of Clusters (k)'); axes[0].set_ylabel('Inertia')
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, marker='s', color='tomato')
axes[1].axvline(x=4, color='red', linestyle='--', label='k=4 chosen')
axes[1].set_title('Silhouette Score by k')
axes[1].set_xlabel('Number of Clusters (k)'); axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nBest k: {list(K_range)[sil_scores.index(max(sil_scores))]}  |  Score: {max(sil_scores):.3f}")

### Observation

The elbow point appears at k=4, and the silhouette score confirms this as the optimal number of customer groups.

## Step 8 — Customer Segmentation Using K-Means (k=4)

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

score = silhouette_score(rfm_scaled, rfm['Cluster'])
print(f"Silhouette Score : {score:.4f}")
print(f"Inertia          : {kmeans.inertia_:.2f}")

# View actual cluster averages — required before assigning labels
cluster_summary = rfm.groupby('Cluster')[['Recency','Frequency','Monetary']].mean().round(1)
print("\nCluster Averages:")
print(cluster_summary)

In [ ]:
# Assign labels based on actual RFM means — exactly matching template labels
# Template labels: High-Value | Regular | Occasional | At-Risk
cs = cluster_summary.copy()
label_map = {}

# High-Value  → highest Monetary (big spenders, frequent, recent)
label_map[cs['Monetary'].idxmax()] = 'High-Value'

# At-Risk     → highest Recency (haven't purchased in longest time)
remaining = [i for i in cs.index if i not in label_map]
label_map[cs.loc[remaining, 'Recency'].idxmax()] = 'At-Risk'

# Regular     → highest Frequency among remaining (steady purchasers)
remaining = [i for i in cs.index if i not in label_map]
label_map[cs.loc[remaining, 'Frequency'].idxmax()] = 'Regular'

# Occasional  → the last remaining cluster (low F, low M)
remaining = [i for i in cs.index if i not in label_map]
label_map[remaining[0]] = 'Occasional'

rfm['Segment'] = rfm['Cluster'].map(label_map)

print("Cluster → Segment mapping:")
for k, v in sorted(label_map.items()):
    row = cluster_summary.loc[k]
    print(f"  Cluster {k} → {v:<12}  (Recency={row['Recency']:.0f}d, Frequency={row['Frequency']:.1f}, Monetary=£{row['Monetary']:.0f})")

print("\nSegment counts:")
print(rfm['Segment'].value_counts())

In [ ]:
# Cluster profile bar chart (Recency & Frequency)
colors_map = {
    'High-Value' : '#F5C400',
    'Regular'    : '#2196F3',
    'Occasional' : '#4CAF50',
    'At-Risk'    : '#F44336'
}

cluster_profile = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(1)
cluster_profile[['Recency','Frequency']].plot(kind='bar', figsize=(10, 5), colormap='Set2', edgecolor='white')
plt.title('Cluster Profiles — Recency and Frequency')
plt.ylabel('Average Value')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# 2D scatter — Frequency vs Monetary
plt.figure(figsize=(10, 6))
for seg, grp in rfm.groupby('Segment'):
    plt.scatter(grp['Frequency'], grp['Monetary'], label=seg,
                color=colors_map[seg], alpha=0.6, s=25)
plt.title('Customer Segmentation — Frequency vs Monetary')
plt.xlabel('Frequency (number of purchases)')
plt.ylabel('Monetary (total spend £)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 3D scatter — Recency, Frequency, Monetary
fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
for seg, grp in rfm.groupby('Segment'):
    ax.scatter(grp['Recency'], grp['Frequency'], grp['Monetary'],
               label=seg, color=colors_map[seg], alpha=0.6, s=20)
ax.set_xlabel('Recency (days)')
ax.set_ylabel('Frequency')
ax.set_zlabel('Monetary (£)')
ax.set_title('Customer Segments — 3D RFM View', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Segment distribution bar chart
seg_counts = rfm['Segment'].value_counts()
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(seg_counts.index, seg_counts.values,
              color=[colors_map[s] for s in seg_counts.index], edgecolor='white')
for bar, val in zip(bars, seg_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, str(val),
            ha='center', fontsize=11, fontweight='bold')
ax.set_title('Customer Segment Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Customers'); ax.set_xlabel('Segment')
plt.tight_layout()
plt.show()

## Cluster Evaluation

A Silhouette Score of **0.616** confirms the K-Means model successfully identified well-separated customer segments.

| Segment | Characteristics | Strategy |
|---|---|---|
| High-Value | Recent, frequent, high spenders | Reward and retain — VIP treatment |
| Regular | Steady purchasers, moderate spend | Loyalty programs and upsell |
| Occasional | Rare, infrequent purchases | Re-engagement campaigns |
| At-Risk | Haven't purchased in a long time | Win-back campaigns, special offers |

## Step 9 — Product Recommendation System

Item-Based Collaborative Filtering using Cosine Similarity identifies products frequently bought by the same customers.

In [ ]:
# Build Customer × Product matrix
customer_product = pd.crosstab(df_clean['CustomerID'], df_clean['Description'])
print(f"Customer-Product matrix: {customer_product.shape}  (Customers × Products)")

In [ ]:
# Compute item-item cosine similarity
similarity_matrix = cosine_similarity(customer_product.T)
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=customer_product.columns,
    columns=customer_product.columns
)
print("Similarity matrix shape:", similarity_df.shape)

In [ ]:
# Recommendation function with partial name matching
def recommend_products(product_name, n=5):
    product_name = product_name.strip().upper()
    if product_name in similarity_df.index:
        match = product_name
    else:
        matches = [p for p in similarity_df.index if product_name in p.upper()]
        if not matches:
            return pd.Series(dtype=float), f"Product not found: {product_name}"
        match = matches[0]
    similar = similarity_df[match].sort_values(ascending=False)[1:n+1]
    return similar, match

In [ ]:
# Demo 1 — show actual recommendations with scores
test_product = '10 COLOUR SPACEBOY PEN'
recs, matched = recommend_products(test_product)

print(f"Input   : {test_product}")
print(f"Matched : {matched}")
print(f"\nTop 5 similar products:")
for i, (prod, score) in enumerate(recs.items(), 1):
    print(f"  {i}. {prod}  (similarity: {score:.3f})")

In [ ]:
# Visualize recommendations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Product Recommendation System', fontsize=13, fontweight='bold')

# Recommendation bar chart
recs, matched = recommend_products('10 COLOUR SPACEBOY PEN')
axes[0].barh(recs.index[::-1], recs.values[::-1], color='royalblue', edgecolor='white')
axes[0].set_title(f'Top 5 Similar Products\nfor: {matched}')
axes[0].set_xlabel('Cosine Similarity Score')
axes[0].set_xlim(0, 1)

# Similarity heatmap of top 12 products
top12 = df_clean.groupby('Description')['Quantity'].sum()        .sort_values(ascending=False).head(12).index
subset = similarity_df.loc[top12, top12]
sns.heatmap(subset, cmap='Blues', ax=axes[1], linewidths=0.5,
            xticklabels=False, yticklabels=True, annot=False,
            cbar_kws={'label':'Similarity'})
axes[1].set_title('Product Similarity Heatmap (Top 12)')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Demo 2 — second product test
recs2, matched2 = recommend_products('WHITE HANGING HEART T-LIGHT HOLDER')
print(f"Recommendations for: {matched2}\n")
for i, (prod, score) in enumerate(recs2.items(), 1):
    print(f"  {i}. {prod}  (similarity: {score:.3f})")

## Step 10 — Save Models

In [ ]:
os.makedirs('models', exist_ok=True)

joblib.dump(kmeans,        'models/kmeans_model.pkl')
joblib.dump(scaler,        'models/scaler.pkl')
joblib.dump(similarity_df, 'models/similarity_matrix.pkl')
rfm.to_csv('models/customer_segments.csv', index=False)

print("✅ Saved: models/kmeans_model.pkl")
print("✅ Saved: models/scaler.pkl")
print("✅ Saved: models/similarity_matrix.pkl")
print("✅ Saved: models/customer_segments.csv")

## Step 11 — Model Evaluation

In [ ]:
final_sil = silhouette_score(rfm_scaled, rfm['Cluster'])

print("=" * 50)
print("       FINAL MODEL EVALUATION REPORT")
print("=" * 50)
print(f"  Algorithm          : KMeans Clustering")
print(f"  Number of Clusters : 4")
print(f"  Silhouette Score   : {final_sil:.4f}  (-1 to 1, higher = better)")
print(f"  Inertia            : {kmeans.inertia_:.2f}")
print(f"  Total Customers    : {len(rfm):,}")
print("=" * 50)
print("\nFinal Cluster Profiles:")
profile = rfm.groupby('Segment')[['Recency','Frequency','Monetary']].mean().round(1)
profile['Customer Count'] = rfm['Segment'].value_counts()
print(profile)

## Step 12 — Business Insights

### Customer Segmentation
- Four customer segments were identified using K-Means clustering
- High-Value customers are few but generate the highest revenue per person
- At-Risk customers have high recency and require immediate re-engagement

### Product Recommendations
- Collaborative filtering identifies products bought by similar customers
- Cosine similarity scores above 0.3 indicate strong product affinity
- Cross-selling recommendations can increase average order value

### Business Impact
- Improved customer targeting through data-driven segmentation
- Personalized product recommendations to increase basket size
- Proactive retention campaigns for At-Risk customers
- Revenue optimization by focusing on High-Value and Regular segments

## Project Conclusion

This project successfully analyzed customer purchasing behavior using **RFM Analysis** and **K-Means Clustering**.

Four customer segments were identified:
- **High-Value** — Recent, frequent, highest spenders
- **Regular** — Steady, consistent buyers
- **Occasional** — Rare, infrequent buyers with growth potential
- **At-Risk** — Inactive customers requiring win-back campaigns

A product recommendation engine was built using **Item-Based Collaborative Filtering** and **Cosine Similarity**.

### Key Achievements

| Metric | Value |
|---|---|
| Total records cleaned | 397,884 |
| Customers segmented | 4,338 |
| Clusters identified | 4 |
| Silhouette Score | 0.616 |
| Products in similarity matrix | 3,877 |

### Future Scope
- Deploy the model using Streamlit web application
- Integrate real-time recommendations using an API
- Use Matrix Factorization (SVD) for improved recommendation accuracy
- Add A/B testing to measure campaign effectiveness per segment